# DPO Training for AI Interview Coach (A100)

This notebook finishes the DPO stage after SFT. It uses the existing preference pairs, downloads the SFT LoRA adapter from Hugging Face, trains a DPO adapter with QLoRA on an A100 GPU, then packages and optionally uploads the result.

SFT adapter used here: `Hank122222222/ai-interview-coach-sft-adapter`.

## 0. Runtime Check

In Colab, choose **Runtime > Change runtime type > A100 GPU** before running the notebook.

In [ ]:
!nvidia-smi

## 1. Clone the Project

This cell clones the GitHub repo into Colab and enters the project directory. If the folder already exists, it updates it.

In [ ]:
import os
from pathlib import Path

PROJECT_PATH = Path('/content/ai-interview-coach')
REPO_URL = 'https://github.com/dcyforjob2020/ai-interview-coach.git'

if PROJECT_PATH.exists():
    %cd /content/ai-interview-coach
    !git fetch origin main
    !git checkout main
    !git pull --ff-only origin main
else:
    !git clone {REPO_URL} {PROJECT_PATH}
    %cd /content/ai-interview-coach

print('Project path:', PROJECT_PATH)

## 2. Install Dependencies

These versions are chosen for the repo scripts and TRL DPO trainer API. If Colab asks to restart after installation, restart the runtime and continue from the next cell.

In [ ]:
!pip install -q -U \
  "transformers==4.46.3" \
  "accelerate>=1.1.0" \
  "peft>=0.13.2" \
  "datasets>=3.1.0" \
  "bitsandbytes>=0.44.1" \
  "trl==0.12.2" \
  "sentencepiece" \
  "protobuf<6" \
  "huggingface_hub"

In [ ]:
import torch
import transformers
import peft
import trl
import bitsandbytes
import datasets

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('trl:', trl.__version__)
print('datasets:', datasets.__version__)
print('bitsandbytes:', bitsandbytes.__version__)

## 3. Login to Hugging Face

Login is needed to download reliably and to upload the final DPO adapter. Use a token with write access if you want to upload the final adapter.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 4. Configuration

For A100, the defaults below should be comfortable for Qwen2.5-3B with QLoRA. If you hit an out-of-memory error, set `PER_DEVICE_BATCH_SIZE = 1`.

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
SFT_ADAPTER_REPO = 'Hank122222222/ai-interview-coach-sft-adapter'
DPO_OUTPUT_DIR = 'train/model/dpo_adapter'
DPO_HF_REPO = 'Hank122222222/ai-interview-coach-dpo-adapter'

MAX_PROMPT_LENGTH = 1536
MAX_LENGTH = 2048
EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
BETA = 0.1
SAVE_STEPS = 100
LOGGING_STEPS = 10

print('Base model:', BASE_MODEL)
print('SFT adapter:', SFT_ADAPTER_REPO)
print('DPO output:', DPO_OUTPUT_DIR)

## 5. Build and Validate the DPO Dataset

This converts `train/preference_pairs.jsonl` into the chat-format DPO dataset expected by the trainer.

In [ ]:
!python train/dpo/build_dpo_dataset.py \
  --input train/preference_pairs.jsonl \
  --output train/dpo/dpo_dataset.jsonl

In [ ]:
import json
from pathlib import Path

path = Path('train/dpo/dpo_dataset.jsonl')
count = 0
first = None
with path.open('r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        record = json.loads(line)
        if first is None:
            first = record
        count += 1

print('DPO records:', count)
print('First record keys:', sorted(first.keys()))
print('First id:', first['id'])
print('Prompt roles:', [m['role'] for m in first['prompt']])
print('Chosen role:', first['chosen'][0]['role'])
print('Rejected role:', first['rejected'][0]['role'])
print('Prompt preview:', first['prompt'][1]['content'][:500])

## 6. Download the SFT Adapter Locally

The repo's DPO script expects a local SFT adapter folder. This cell downloads the Hugging Face SFT adapter into that exact path.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

local_sft_adapter = Path('train/model/sft_adapter')
local_sft_adapter.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=SFT_ADAPTER_REPO,
    repo_type='model',
    local_dir=str(local_sft_adapter),
    local_dir_use_symlinks=False,
)

print('Downloaded SFT adapter to:', local_sft_adapter)
print('Adapter files:')
for p in sorted(local_sft_adapter.iterdir()):
    print(' -', p.name)

## 7. Train DPO with QLoRA

This is the main training cell. On an A100, `max_length=2048` and batch size 2 should usually fit. Training starts from the SFT adapter and saves the DPO adapter to `train/model/dpo_adapter`.

In [ ]:
!python train/dpo/train_dpo_qlora.py \
  --model {BASE_MODEL} \
  --data train/dpo/dpo_dataset.jsonl \
  --sft-adapter train/model/sft_adapter \
  --output-dir {DPO_OUTPUT_DIR} \
  --max-prompt-length {MAX_PROMPT_LENGTH} \
  --max-length {MAX_LENGTH} \
  --epochs {EPOCHS} \
  --batch-size {PER_DEVICE_BATCH_SIZE} \
  --grad-accum {GRAD_ACCUM} \
  --learning-rate {LEARNING_RATE} \
  --beta {BETA} \
  --save-steps {SAVE_STEPS} \
  --logging-steps {LOGGING_STEPS}

## 8. Check the Saved DPO Adapter

This confirms the adapter files exist after training.

In [ ]:
## 7. Train DPO with QLoRA

This is the main training cell. It loads the SFT LoRA adapter twice: the `default` adapter is trainable as the DPO policy, and the `reference` adapter stays frozen as the DPO reference model. On an A100, `max_length=2048` and batch size 2 should usually fit.

import gc
from pathlib import Path

import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

gc.collect()
torch.cuda.empty_cache()
torch.backends.cuda.matmul.allow_tf32 = True

local_sft_adapter = 'train/model/sft_adapter'
Path(DPO_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(local_sft_adapter, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

dataset = load_dataset('json', data_files='train/dpo/dpo_dataset.jsonl', split='train')
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map='auto',
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    base_model,
    local_sft_adapter,
    adapter_name='default',
    is_trainable=True,
)
model.load_adapter(
    local_sft_adapter,
    adapter_name='reference',
    is_trainable=False,
)
model.set_adapter('default')
model.config.use_cache = False

training_args = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    beta=BETA,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_length=MAX_LENGTH,
    model_adapter_name='default',
    ref_adapter_name='reference',
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

# Save only the trained DPO policy adapter, not the frozen reference copy.
model.set_adapter('default')
try:
    model.save_pretrained(DPO_OUTPUT_DIR, selected_adapters=['default'])
except TypeError:
    if 'reference' in getattr(model, 'peft_config', {}):
        model.delete_adapter('reference')
    model.set_adapter('default')
    model.save_pretrained(DPO_OUTPUT_DIR)
tokenizer.save_pretrained(DPO_OUTPUT_DIR)

print('Saved DPO adapter to:', DPO_OUTPUT_DIR)

del trainer, model, base_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import gc
import json
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(DPO_OUTPUT_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map='auto',
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, DPO_OUTPUT_DIR)
model.eval()

with open('data/test.jsonl', 'r', encoding='utf-8') as f:
    example = json.loads(next(f))

messages = [
    {
        'role': 'system',
        'content': 'You are a precise and helpful AI interview coach for CS students. Give feedback directly to the student using second person. Identify what is correct, what is missing or incorrect, and how to improve the answer.',
    },
    {
        'role': 'user',
        'content': f'''Question:
{example['question']}

Reference answer:
{example['reference_answer']}

Student answer:
{example['student_answer']}

Write interview coaching feedback for the student.''',
    },
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0, inputs['input_ids'].shape[-1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

## 10. Package the DPO Adapter

This creates a zip you can download from Colab even if you also upload to Hugging Face.

In [ ]:
!zip -r dpo_adapter_final_only.zip \
  train/model/dpo_adapter/adapter_config.json \
  train/model/dpo_adapter/adapter_model.safetensors \
  train/model/dpo_adapter/tokenizer.json \
  train/model/dpo_adapter/tokenizer_config.json \
  train/model/dpo_adapter/special_tokens_map.json \
  train/model/dpo_adapter/vocab.json \
  train/model/dpo_adapter/merges.txt \
  train/model/dpo_adapter/README.md \
  train/model/dpo_adapter/added_tokens.json \
  train/model/dpo_adapter/training_args.bin

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

zip_path = Path('dpo_adapter_final_only.zip')
adapter_dir = Path(DPO_OUTPUT_DIR)
preferred_files = [
    'adapter_config.json',
    'adapter_model.safetensors',
    'tokenizer.json',
    'tokenizer_config.json',
    'special_tokens_map.json',
    'vocab.json',
    'merges.txt',
    'README.md',
    'added_tokens.json',
    'training_args.bin',
]

with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as zf:
    for name in preferred_files:
        file_path = adapter_dir / name
        if file_path.exists():
            zf.write(file_path, arcname=str(file_path))
            print('added:', file_path)
        else:
            print('skipped missing optional file:', file_path)

print('Created:', zip_path, zip_path.stat().st_size, 'bytes')

## 11. Upload the DPO Adapter to Hugging Face

Run this cell if you want the final DPO adapter saved on Hugging Face. It uploads only final adapter files and ignores checkpoints/optimizer state.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(
    repo_id=DPO_HF_REPO,
    repo_type='model',
    private=False,
    exist_ok=True,
)

api.upload_folder(
    folder_path=DPO_OUTPUT_DIR,
    repo_id=DPO_HF_REPO,
    repo_type='model',
    commit_message='Upload DPO LoRA adapter',
    ignore_patterns=[
        'checkpoint-*',
        'optimizer.pt',
        'scheduler.pt',
        'rng_state.pth',
        'trainer_state.json',
    ],
)

print('Uploaded to:', f'https://huggingface.co/{DPO_HF_REPO}')

## 12. Final Notes

After this notebook finishes, the next project step is generating feedback from the DPO adapter on the fixed test set, scoring it with the judge model, and comparing against the baseline scores.